## Assignment -4: U24AI038 NLP

1. Take the data you tokenized (your language data) in Assignment-1 and build 4 language
model as the following:<br>
a. Unigram Model<br>
b. Bigram Model<br>
c. Trigram Model<br>
d. Quadrigram Model<br>

Take at least 1000000 sentences and split them into training, development, and test sets.
Development and test sets have 1000 sentences each, and training data contains rest of the
sentences.

In [28]:
from pathlib import Path
from collections import Counter

word_tokens = Path("../assignment 1/tokenized_words.txt")

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
quadgram_counts = Counter()

sentence_count = 0

with word_tokens.open("r", encoding="utf-8") as f:
    for sentence in f:

        tokens = sentence.split()
        if not tokens:
            continue

        # Sentence boundaries
        tokens = ["<s>"] + tokens + ["</s>"]

        # Unigrams
        unigram_counts.update(tokens)

        # Bigrams
        bigram_counts.update(
            zip(tokens, tokens[1:])
        )

        # Trigrams
        trigram_counts.update(
            zip(tokens, tokens[1:], tokens[2:])
        )

        # Quadrigrams
        quadgram_counts.update(
            zip(tokens, tokens[1:], tokens[2:], tokens[3:])
        )

        sentence_count += 1

        if sentence_count == 100:
            break

print("Sentences:", sentence_count)
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))
print("Unique quadrigrams:", len(quadgram_counts))

Sentences: 100
Unique unigrams: 1030
Unique bigrams: 1486
Unique trigrams: 1522
Unique quadrigrams: 1482


In [29]:
from pathlib import Path
import duckdb
import pyarrow as pa
import time


INPUT_FILE = Path("../assignment 1/tokenized_words.txt")
DB_FILE = Path("ngram_model.duckdb")

DEV_FILE = Path("dev.txt")
TEST_FILE = Path("test.txt")

TRAIN_SENTENCES = 1_000_000
DEV_SENTENCES = 1_000
TEST_SENTENCES = 1_000

BATCH_SIZE = 5_000


# Start with a fresh database
try:
    con.close()
except:
    pass

if DB_FILE.exists():
    DB_FILE.unlink()

con = duckdb.connect(str(DB_FILE))
con.execute("PRAGMA threads=8")


# Create n-gram tables
con.execute("""
CREATE TABLE unigrams (
    token TEXT PRIMARY KEY,
    count BIGINT
)
""")

con.execute("""
CREATE TABLE bigrams (
    w1 TEXT,
    w2 TEXT,
    count BIGINT,
    PRIMARY KEY (w1, w2)
)
""")

con.execute("""
CREATE TABLE trigrams (
    w1 TEXT,
    w2 TEXT,
    w3 TEXT,
    count BIGINT,
    PRIMARY KEY (w1, w2, w3)
)
""")

con.execute("""
CREATE TABLE quadrigrams (
    w1 TEXT,
    w2 TEXT,
    w3 TEXT,
    w4 TEXT,
    count BIGINT,
    PRIMARY KEY (w1, w2, w3, w4)
)
""")


def process_batch(batch):

    unigrams = []
    bigrams = []
    trigrams = []
    quadrigrams = []

    for tokens in batch:

        if not tokens:
            continue

        tokens = ["<s>"] + tokens + ["</s>"]

        unigrams.extend(tokens)

        bigrams.extend(
            (tokens[i], tokens[i + 1])
            for i in range(len(tokens) - 1)
        )

        trigrams.extend(
            (tokens[i], tokens[i + 1], tokens[i + 2])
            for i in range(len(tokens) - 2)
        )

        quadrigrams.extend(
            (
                tokens[i],
                tokens[i + 1],
                tokens[i + 2],
                tokens[i + 3]
            )
            for i in range(len(tokens) - 3)
        )

    # Convert batches to Arrow tables
    uni_table = pa.table({
        "token": unigrams
    })

    bi_table = pa.table({
        "w1": [x[0] for x in bigrams],
        "w2": [x[1] for x in bigrams]
    })

    tri_table = pa.table({
        "w1": [x[0] for x in trigrams],
        "w2": [x[1] for x in trigrams],
        "w3": [x[2] for x in trigrams]
    })

    quad_table = pa.table({
        "w1": [x[0] for x in quadrigrams],
        "w2": [x[1] for x in quadrigrams],
        "w3": [x[2] for x in quadrigrams],
        "w4": [x[3] for x in quadrigrams]
    })

    con.register("batch_uni", uni_table)
    con.register("batch_bi", bi_table)
    con.register("batch_tri", tri_table)
    con.register("batch_quad", quad_table)

    # Update unigram counts
    con.execute("""
        INSERT INTO unigrams
        SELECT token, COUNT(*)
        FROM batch_uni
        GROUP BY token
        ON CONFLICT (token)
        DO UPDATE SET count =
            unigrams.count + EXCLUDED.count
    """)

    # Update bigram counts
    con.execute("""
        INSERT INTO bigrams
        SELECT w1, w2, COUNT(*)
        FROM batch_bi
        GROUP BY w1, w2
        ON CONFLICT (w1, w2)
        DO UPDATE SET count =
            bigrams.count + EXCLUDED.count
    """)

    # Update trigram counts
    con.execute("""
        INSERT INTO trigrams
        SELECT w1, w2, w3, COUNT(*)
        FROM batch_tri
        GROUP BY w1, w2, w3
        ON CONFLICT (w1, w2, w3)
        DO UPDATE SET count =
            trigrams.count + EXCLUDED.count
    """)

    # Update quadrigram counts
    con.execute("""
        INSERT INTO quadrigrams
        SELECT w1, w2, w3, w4, COUNT(*)
        FROM batch_quad
        GROUP BY w1, w2, w3, w4
        ON CONFLICT (w1, w2, w3, w4)
        DO UPDATE SET count =
            quadrigrams.count + EXCLUDED.count
    """)

    con.unregister("batch_uni")
    con.unregister("batch_bi")
    con.unregister("batch_tri")
    con.unregister("batch_quad")


start_time = time.time()

train_count = 0
dev_count = 0
test_count = 0

batch = []

dev_out = DEV_FILE.open("w", encoding="utf-8")
test_out = TEST_FILE.open("w", encoding="utf-8")


with INPUT_FILE.open("r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        if train_count < TRAIN_SENTENCES:

            batch.append(line.split())
            train_count += 1

            if len(batch) >= BATCH_SIZE:

                process_batch(batch)
                batch.clear()

                elapsed = time.time() - start_time

                print(
                    f"Training: "
                    f"{train_count:,}/{TRAIN_SENTENCES:,} "
                    f"({train_count / TRAIN_SENTENCES * 100:.1f}%) "
                    f"| {elapsed / 60:.2f} min"
                )

        elif dev_count < DEV_SENTENCES:

            dev_out.write(line + "\n")
            dev_count += 1

        elif test_count < TEST_SENTENCES:

            test_out.write(line + "\n")
            test_count += 1

        else:
            break


if batch:
    process_batch(batch)
    batch.clear()


dev_out.close()
test_out.close()

con.commit()


elapsed = time.time() - start_time


# Final statistics
unique_unigrams = con.execute(
    "SELECT COUNT(*) FROM unigrams"
).fetchone()[0]

unique_bigrams = con.execute(
    "SELECT COUNT(*) FROM bigrams"
).fetchone()[0]

unique_trigrams = con.execute(
    "SELECT COUNT(*) FROM trigrams"
).fetchone()[0]

unique_quadrigrams = con.execute(
    "SELECT COUNT(*) FROM quadrigrams"
).fetchone()[0]


total_unigrams = con.execute(
    "SELECT SUM(count) FROM unigrams"
).fetchone()[0]

total_bigrams = con.execute(
    "SELECT SUM(count) FROM bigrams"
).fetchone()[0]

total_trigrams = con.execute(
    "SELECT SUM(count) FROM trigrams"
).fetchone()[0]

total_quadrigrams = con.execute(
    "SELECT SUM(count) FROM quadrigrams"
).fetchone()[0]


db_size = DB_FILE.stat().st_size / (1024 ** 3)


print()
print("Processing complete")
print("-------------------")

print(f"Training sentences : {train_count:,}")
print(f"Development        : {dev_count:,}")
print(f"Test               : {test_count:,}")

print()
print(f"Unique unigrams    : {unique_unigrams:,}")
print(f"Unique bigrams     : {unique_bigrams:,}")
print(f"Unique trigrams    : {unique_trigrams:,}")
print(f"Unique quadrigrams : {unique_quadrigrams:,}")

print()
print(f"Total unigram tokens    : {total_unigrams:,}")
print(f"Total bigram tokens     : {total_bigrams:,}")
print(f"Total trigram tokens    : {total_trigrams:,}")
print(f"Total quadrigram tokens : {total_quadrigrams:,}")

print()
print(f"Time taken : {elapsed / 60:.2f} minutes")
print(f"Database   : {db_size:.2f} GB")

con.close()

Training: 5,000/1,000,000 (0.5%) | 0.01 min
Training: 10,000/1,000,000 (1.0%) | 0.04 min
Training: 15,000/1,000,000 (1.5%) | 0.06 min
Training: 20,000/1,000,000 (2.0%) | 0.08 min
Training: 25,000/1,000,000 (2.5%) | 0.11 min
Training: 30,000/1,000,000 (3.0%) | 0.14 min
Training: 35,000/1,000,000 (3.5%) | 0.17 min
Training: 40,000/1,000,000 (4.0%) | 0.20 min
Training: 45,000/1,000,000 (4.5%) | 0.23 min
Training: 50,000/1,000,000 (5.0%) | 0.26 min
Training: 55,000/1,000,000 (5.5%) | 0.30 min
Training: 60,000/1,000,000 (6.0%) | 0.34 min
Training: 65,000/1,000,000 (6.5%) | 0.37 min
Training: 70,000/1,000,000 (7.0%) | 0.41 min
Training: 75,000/1,000,000 (7.5%) | 0.46 min
Training: 80,000/1,000,000 (8.0%) | 0.51 min
Training: 85,000/1,000,000 (8.5%) | 0.56 min
Training: 90,000/1,000,000 (9.0%) | 0.62 min
Training: 95,000/1,000,000 (9.5%) | 0.67 min
Training: 100,000/1,000,000 (10.0%) | 0.73 min
Training: 105,000/1,000,000 (10.5%) | 0.79 min
Training: 110,000/1,000,000 (11.0%) | 0.85 min
Train

In [30]:


con = duckdb.connect("ngram_model.duckdb", read_only=True)

result = con.execute("""
SELECT
    (SELECT SUM(count) FROM unigrams) AS total_unigrams,
    (SELECT SUM(count) FROM bigrams) AS total_bigrams,
    (SELECT SUM(count) FROM trigrams) AS total_trigrams,
    (SELECT SUM(count) FROM quadrigrams) AS total_quadrigrams
""").fetchone()

print("Total unigrams    :", result[0])
print("Total bigrams     :", result[1])
print("Total trigrams    :", result[2])
print("Total quadrigrams :", result[3])

con.close()

Total unigrams    : 17644175
Total bigrams     : 16644175
Total trigrams    : 15644175
Total quadrigrams : 14644175


In [31]:
from pathlib import Path
import duckdb
import math
import time


DB_FILE = Path("ngram_model.duckdb")
DEV_FILE = Path("dev.txt")
TEST_FILE = Path("test.txt")


con = duckdb.connect(str(DB_FILE), read_only=True)


# Load counts from the database

unigram_counts = {
    row[0]: row[1]
    for row in con.execute("""
        SELECT token, count
        FROM unigrams
    """).fetchall()
}

bigram_counts = {
    (row[0], row[1]): row[2]
    for row in con.execute("""
        SELECT w1, w2, count
        FROM bigrams
    """).fetchall()
}

trigram_counts = {
    (row[0], row[1], row[2]): row[3]
    for row in con.execute("""
        SELECT w1, w2, w3, count
        FROM trigrams
    """).fetchall()
}

quadrigram_counts = {
    (row[0], row[1], row[2], row[3]): row[4]
    for row in con.execute("""
        SELECT w1, w2, w3, w4, count
        FROM quadrigrams
    """).fetchall()
}


con.close()


total_unigrams = sum(unigram_counts.values())


print("Counts loaded")
print("----------------")
print(f"Unigrams    : {len(unigram_counts):,}")
print(f"Bigrams     : {len(bigram_counts):,}")
print(f"Trigrams    : {len(trigram_counts):,}")
print(f"Quadrigrams : {len(quadrigram_counts):,}")
print(f"Total tokens: {total_unigrams:,}")


# Calculate sentence log probability

def unigram_log_probability(tokens):

    log_probability = 0.0

    for word in tokens:

        count = unigram_counts.get(word, 0)

        if count == 0:
            return float("-inf")

        probability = count / total_unigrams

        log_probability += math.log(probability)

    return log_probability


def bigram_log_probability(tokens):

    log_probability = 0.0

    for i in range(len(tokens) - 1):

        w1 = tokens[i]
        w2 = tokens[i + 1]

        count = bigram_counts.get((w1, w2), 0)

        denominator = unigram_counts.get(w1, 0)

        if count == 0 or denominator == 0:
            return float("-inf")

        probability = count / denominator

        log_probability += math.log(probability)

    return log_probability


def trigram_log_probability(tokens):

    log_probability = 0.0

    for i in range(len(tokens) - 2):

        w1 = tokens[i]
        w2 = tokens[i + 1]
        w3 = tokens[i + 2]

        count = trigram_counts.get(
            (w1, w2, w3),
            0
        )

        denominator = bigram_counts.get(
            (w1, w2),
            0
        )

        if count == 0 or denominator == 0:
            return float("-inf")

        probability = count / denominator

        log_probability += math.log(probability)

    return log_probability


def quadrigram_log_probability(tokens):

    log_probability = 0.0

    for i in range(len(tokens) - 3):

        w1 = tokens[i]
        w2 = tokens[i + 1]
        w3 = tokens[i + 2]
        w4 = tokens[i + 3]

        count = quadrigram_counts.get(
            (w1, w2, w3, w4),
            0
        )

        denominator = trigram_counts.get(
            (w1, w2, w3),
            0
        )

        if count == 0 or denominator == 0:
            return float("-inf")

        probability = count / denominator

        log_probability += math.log(probability)

    return log_probability


def prepare_sentence(sentence):

    tokens = sentence.split()

    if not tokens:
        return []

    return ["<s>"] + tokens + ["</s>"]

Counts loaded
----------------
Unigrams    : 762,304
Bigrams     : 6,071,268
Trigrams    : 10,837,619
Quadrigrams : 12,512,961
Total tokens: 17,644,175


In [34]:
with DEV_FILE.open("r", encoding="utf-8") as f:
    sentence = next(f).strip()

tokens = prepare_sentence(sentence)

print("Sentence:")
print(sentence)

print("\nTokens:")
print(tokens)

print("\nLog probabilities:")

print(
    "Unigram    :",
    unigram_log_probability(tokens)
)

print(
    "Bigram     :",
    bigram_log_probability(tokens)
)

print(
    "Trigram    :",
    trigram_log_probability(tokens)
)

print(
    "Quadrigram :",
    quadrigram_log_probability(tokens)
)

Sentence:
રાકેશ ટીકેતે કહ્યું, “તમામ વ્યવસ્થા કરવામાં આવી છે .

Tokens:
['<s>', 'રાકેશ', 'ટીકેતે', 'કહ્યું,', '“તમામ', 'વ્યવસ્થા', 'કરવામાં', 'આવી', 'છે', '.', '</s>']

Log probabilities:
Unigram    : -inf
Bigram     : -inf
Trigram    : -inf
Quadrigram : -inf


In [35]:
tokens = prepare_sentence(sentence)

print("UNIGRAMS")
for w in tokens:
    count = unigram_counts.get(w, 0)
    print(f"{w:20} {count}")


print("\nBIGRAMS")
for i in range(len(tokens) - 1):
    bg = (tokens[i], tokens[i + 1])
    count = bigram_counts.get(bg, 0)
    print(f"{bg} -> {count}")


print("\nTRIGRAMS")
for i in range(len(tokens) - 2):
    tg = (
        tokens[i],
        tokens[i + 1],
        tokens[i + 2]
    )
    count = trigram_counts.get(tg, 0)
    print(f"{tg} -> {count}")


print("\nQUADRIGRAMS")
for i in range(len(tokens) - 3):
    qg = (
        tokens[i],
        tokens[i + 1],
        tokens[i + 2],
        tokens[i + 3]
    )
    count = quadrigram_counts.get(qg, 0)
    print(f"{qg} -> {count}")

UNIGRAMS
<s>                  1000000
રાકેશ                537
ટીકેતે               0
કહ્યું,              1932
“તમામ                5
વ્યવસ્થા             3077
કરવામાં              49872
આવી                  49924
છે                   548088
.                    850735
</s>                 1000000

BIGRAMS
('<s>', 'રાકેશ') -> 94
('રાકેશ', 'ટીકેતે') -> 0
('ટીકેતે', 'કહ્યું,') -> 0
('કહ્યું,', '“તમામ') -> 0
('“તમામ', 'વ્યવસ્થા') -> 0
('વ્યવસ્થા', 'કરવામાં') -> 367
('કરવામાં', 'આવી') -> 14149
('આવી', 'છે') -> 11926
('છે', '.') -> 385054
('.', '</s>') -> 850735

TRIGRAMS
('<s>', 'રાકેશ', 'ટીકેતે') -> 0
('રાકેશ', 'ટીકેતે', 'કહ્યું,') -> 0
('ટીકેતે', 'કહ્યું,', '“તમામ') -> 0
('કહ્યું,', '“તમામ', 'વ્યવસ્થા') -> 0
('“તમામ', 'વ્યવસ્થા', 'કરવામાં') -> 0
('વ્યવસ્થા', 'કરવામાં', 'આવી') -> 260
('કરવામાં', 'આવી', 'છે') -> 4735
('આવી', 'છે', '.') -> 10145
('છે', '.', '</s>') -> 385054

QUADRIGRAMS
('<s>', 'રાકેશ', 'ટીકેતે', 'કહ્યું,') -> 0
('રાકેશ', 'ટીકેતે', 'કહ્યું,', '“તમામ') -> 0
('ટીકેતે', 'કહ્

In [36]:
from pathlib import Path
import duckdb
import math


DB_FILE = Path("ngram_model.duckdb")
DEV_FILE = Path("dev.txt")
TEST_FILE = Path("test.txt")


con = duckdb.connect(str(DB_FILE), read_only=True)

unigram_counts = {
    row[0]: row[1]
    for row in con.execute(
        "SELECT token, count FROM unigrams"
    ).fetchall()
}

bigram_counts = {
    (row[0], row[1]): row[2]
    for row in con.execute(
        "SELECT w1, w2, count FROM bigrams"
    ).fetchall()
}

trigram_counts = {
    (row[0], row[1], row[2]): row[3]
    for row in con.execute(
        "SELECT w1, w2, w3, count FROM trigrams"
    ).fetchall()
}

quadrigram_counts = {
    (row[0], row[1], row[2], row[3]): row[4]
    for row in con.execute(
        "SELECT w1, w2, w3, w4, count FROM quadrigrams"
    ).fetchall()
}

con.close()


total_unigrams = sum(unigram_counts.values())


def prepare_sentence(sentence):
    return ["<s>"] + sentence.strip().split() + ["</s>"]


def unigram_log_probability(tokens):
    log_prob = 0.0

    for token in tokens:
        count = unigram_counts.get(token, 0)

        if count == 0:
            return -math.inf

        log_prob += math.log(count / total_unigrams)

    return log_prob


def bigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(1, len(tokens)):
        w1 = tokens[i - 1]
        w2 = tokens[i]

        numerator = bigram_counts.get((w1, w2), 0)
        denominator = unigram_counts.get(w1, 0)

        if numerator == 0 or denominator == 0:
            return -math.inf

        log_prob += math.log(numerator / denominator)

    return log_prob


def trigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(2, len(tokens)):
        w1 = tokens[i - 2]
        w2 = tokens[i - 1]
        w3 = tokens[i]

        numerator = trigram_counts.get((w1, w2, w3), 0)
        denominator = bigram_counts.get((w1, w2), 0)

        if numerator == 0 or denominator == 0:
            return -math.inf

        log_prob += math.log(numerator / denominator)

    return log_prob


def quadrigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(3, len(tokens)):
        w1 = tokens[i - 3]
        w2 = tokens[i - 2]
        w3 = tokens[i - 1]
        w4 = tokens[i]

        numerator = quadrigram_counts.get((w1, w2, w3, w4), 0)
        denominator = trigram_counts.get((w1, w2, w3), 0)

        if numerator == 0 or denominator == 0:
            return -math.inf

        log_prob += math.log(numerator / denominator)

    return log_prob


def evaluate_file(file_path):
    sentences = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                sentences.append(prepare_sentence(line))

    results = {
        "sentences": len(sentences),
        "oov_words": 0,
        "sentences_with_oov": 0,
        "unseen_bigrams": 0,
        "sentences_with_unseen_bigrams": 0,
        "unseen_trigrams": 0,
        "sentences_with_unseen_trigrams": 0,
        "unseen_quadrigrams": 0,
        "sentences_with_unseen_quadrigrams": 0,
        "unigram_log_probs": [],
        "bigram_log_probs": [],
        "trigram_log_probs": [],
        "quadrigram_log_probs": []
    }

    for tokens in sentences:
        words = tokens[1:-1]

        oov_count = sum(
            1 for word in words
            if unigram_counts.get(word, 0) == 0
        )

        if oov_count > 0:
            results["sentences_with_oov"] += 1
            results["oov_words"] += oov_count

        unseen_bi = any(
            bigram_counts.get((tokens[i - 1], tokens[i]), 0) == 0
            for i in range(1, len(tokens))
        )

        if unseen_bi:
            results["unseen_bigrams"] += sum(
                1
                for i in range(1, len(tokens))
                if bigram_counts.get((tokens[i - 1], tokens[i]), 0) == 0
            )
            results["sentences_with_unseen_bigrams"] += 1

        unseen_tri = any(
            trigram_counts.get(
                (tokens[i - 2], tokens[i - 1], tokens[i]), 0
            ) == 0
            for i in range(2, len(tokens))
        )

        if unseen_tri:
            results["unseen_trigrams"] += sum(
                1
                for i in range(2, len(tokens))
                if trigram_counts.get(
                    (tokens[i - 2], tokens[i - 1], tokens[i]), 0
                ) == 0
            )
            results["sentences_with_unseen_trigrams"] += 1

        unseen_quad = any(
            quadrigram_counts.get(
                (
                    tokens[i - 3],
                    tokens[i - 2],
                    tokens[i - 1],
                    tokens[i]
                ),
                0
            ) == 0
            for i in range(3, len(tokens))
        )

        if unseen_quad:
            results["unseen_quadrigrams"] += sum(
                1
                for i in range(3, len(tokens))
                if quadrigram_counts.get(
                    (
                        tokens[i - 3],
                        tokens[i - 2],
                        tokens[i - 1],
                        tokens[i]
                    ),
                    0
                ) == 0
            )
            results["sentences_with_unseen_quadrigrams"] += 1

        log_probs = [
            unigram_log_probability(tokens),
            bigram_log_probability(tokens),
            trigram_log_probability(tokens),
            quadrigram_log_probability(tokens)
        ]

        results["unigram_log_probs"].append(log_probs[0])
        results["bigram_log_probs"].append(log_probs[1])
        results["trigram_log_probs"].append(log_probs[2])
        results["quadrigram_log_probs"].append(log_probs[3])

    return results


def summarize_model(log_probs):
    finite = [
        value for value in log_probs
        if math.isfinite(value)
    ]

    zero_probability = len(log_probs) - len(finite)

    if not finite:
        return {
            "finite_sentences": 0,
            "zero_probability_sentences": zero_probability,
            "average_log_probability": None,
            "perplexity": None
        }

    avg_log_prob = sum(finite) / len(finite)

    perplexity = math.exp(-avg_log_prob)

    return {
        "finite_sentences": len(finite),
        "zero_probability_sentences": zero_probability,
        "average_log_probability": avg_log_prob,
        "perplexity": perplexity
    }


def print_report(name, results):
    print(f"\n{name}")
    print("-" * len(name))

    print(f"Sentences evaluated: {results['sentences']}")

    print(f"OOV words: {results['oov_words']}")
    print(f"Sentences with OOV: {results['sentences_with_oov']}")

    print(f"Unseen bigrams: {results['unseen_bigrams']}")
    print(
        f"Sentences with unseen bigrams: "
        f"{results['sentences_with_unseen_bigrams']}"
    )

    print(f"Unseen trigrams: {results['unseen_trigrams']}")
    print(
        f"Sentences with unseen trigrams: "
        f"{results['sentences_with_unseen_trigrams']}"
    )

    print(f"Unseen quadrigrams: {results['unseen_quadrigrams']}")
    print(
        f"Sentences with unseen quadrigrams: "
        f"{results['sentences_with_unseen_quadrigrams']}"
    )

    models = {
        "Unigram": results["unigram_log_probs"],
        "Bigram": results["bigram_log_probs"],
        "Trigram": results["trigram_log_probs"],
        "Quadrigram": results["quadrigram_log_probs"]
    }

    print("\nModel results:")

    for model_name, log_probs in models.items():
        summary = summarize_model(log_probs)

        print(f"\n{model_name}")
        print(
            f"  Finite sentences: "
            f"{summary['finite_sentences']}"
        )
        print(
            f"  Zero-probability sentences: "
            f"{summary['zero_probability_sentences']}"
        )

        if summary["average_log_probability"] is not None:
            print(
                f"  Average log probability: "
                f"{summary['average_log_probability']:.4f}"
            )
            print(
                f"  Perplexity: "
                f"{summary['perplexity']:.4f}"
            )
        else:
            print("  Average log probability: undefined")
            print("  Perplexity: undefined")


dev_results = evaluate_file(DEV_FILE)
test_results = evaluate_file(TEST_FILE)

print("=" * 60)
print("MLE EVALUATION REPORT")
print("=" * 60)

print("\nDEVELOPMENT SET")
print_report("Development Set", dev_results)

print("\n" + "=" * 60)

print("\nTEST SET")
print_report("Test Set", test_results)

MLE EVALUATION REPORT

DEVELOPMENT SET

Development Set
---------------
Sentences evaluated: 1000
OOV words: 499
Sentences with OOV: 311
Unseen bigrams: 4996
Sentences with unseen bigrams: 930
Unseen trigrams: 10140
Sentences with unseen trigrams: 983
Unseen quadrigrams: 12355
Sentences with unseen quadrigrams: 984

Model results:

Unigram
  Finite sentences: 689
  Zero-probability sentences: 311
  Average log probability: -129.3715
  Perplexity: 153229086547710146522063310941317008224685723337040592896.0000

Bigram
  Finite sentences: 70
  Zero-probability sentences: 930
  Average log probability: -41.8938
  Perplexity: 1564086437120989440.0000

Trigram
  Finite sentences: 17
  Zero-probability sentences: 983
  Average log probability: -14.0958
  Perplexity: 1323557.0921

Quadrigram
  Finite sentences: 16
  Zero-probability sentences: 984
  Average log probability: -3.7678
  Perplexity: 43.2831


TEST SET

Test Set
--------
Sentences evaluated: 1000
OOV words: 555
Sentences with OOV: 

### 2. Apply Add one/Laplace Smoothing

In [1]:
from pathlib import Path
import duckdb
import math


DB_FILE = Path("ngram_model.duckdb")
DEV_FILE = Path("dev.txt")
TEST_FILE = Path("test.txt")


con = duckdb.connect(str(DB_FILE), read_only=True)

unigram_counts = {
    row[0]: row[1]
    for row in con.execute(
        "SELECT token, count FROM unigrams"
    ).fetchall()
}

bigram_counts = {
    (row[0], row[1]): row[2]
    for row in con.execute(
        "SELECT w1, w2, count FROM bigrams"
    ).fetchall()
}

trigram_counts = {
    (row[0], row[1], row[2]): row[3]
    for row in con.execute(
        "SELECT w1, w2, w3, count FROM trigrams"
    ).fetchall()
}

quadrigram_counts = {
    (row[0], row[1], row[2], row[3]): row[4]
    for row in con.execute(
        "SELECT w1, w2, w3, w4, count FROM quadrigrams"
    ).fetchall()
}

con.close()


vocab_size = len(unigram_counts)
total_unigrams = sum(unigram_counts.values())


def prepare_sentence(sentence):
    return ["<s>"] + sentence.strip().split() + ["</s>"]


def unigram_log_probability(tokens):
    log_prob = 0.0

    denominator = total_unigrams + vocab_size

    for token in tokens:
        count = unigram_counts.get(token, 0)

        probability = (count + 1) / denominator

        log_prob += math.log(probability)

    return log_prob


def bigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(1, len(tokens)):
        w1 = tokens[i - 1]
        w2 = tokens[i]

        numerator = bigram_counts.get((w1, w2), 0) + 1
        denominator = unigram_counts.get(w1, 0) + vocab_size

        probability = numerator / denominator

        log_prob += math.log(probability)

    return log_prob


def trigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(2, len(tokens)):
        w1 = tokens[i - 2]
        w2 = tokens[i - 1]
        w3 = tokens[i]

        numerator = trigram_counts.get((w1, w2, w3), 0) + 1
        denominator = (
            bigram_counts.get((w1, w2), 0) + vocab_size
        )

        probability = numerator / denominator

        log_prob += math.log(probability)

    return log_prob


def quadrigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(3, len(tokens)):
        w1 = tokens[i - 3]
        w2 = tokens[i - 2]
        w3 = tokens[i - 1]
        w4 = tokens[i]

        numerator = (
            quadrigram_counts.get(
                (w1, w2, w3, w4), 0
            ) + 1
        )

        denominator = (
            trigram_counts.get(
                (w1, w2, w3), 0
            ) + vocab_size
        )

        probability = numerator / denominator

        log_prob += math.log(probability)

    return log_prob


def evaluate_file(file_path):
    sentences = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                sentences.append(
                    prepare_sentence(line)
                )

    results = {
        "sentences": len(sentences),
        "unigram_log_probs": [],
        "bigram_log_probs": [],
        "trigram_log_probs": [],
        "quadrigram_log_probs": []
    }

    for tokens in sentences:
        results["unigram_log_probs"].append(
            unigram_log_probability(tokens)
        )

        results["bigram_log_probs"].append(
            bigram_log_probability(tokens)
        )

        results["trigram_log_probs"].append(
            trigram_log_probability(tokens)
        )

        results["quadrigram_log_probs"].append(
            quadrigram_log_probability(tokens)
        )

    return results


def summarize_model(log_probs):
    average_log_prob = sum(log_probs) / len(log_probs)

    perplexity = math.exp(-average_log_prob)

    return {
        "average_log_probability": average_log_prob,
        "perplexity": perplexity
    }


def print_report(name, results):
    print(f"\n{name}")
    print("-" * len(name))

    print(f"Sentences evaluated: {results['sentences']}")

    models = {
        "Unigram": results["unigram_log_probs"],
        "Bigram": results["bigram_log_probs"],
        "Trigram": results["trigram_log_probs"],
        "Quadrigram": results["quadrigram_log_probs"]
    }

    for model_name, log_probs in models.items():
        summary = summarize_model(log_probs)

        print(f"\n{model_name}")
        print(
            f"  Average log probability: "
            f"{summary['average_log_probability']:.4f}"
        )
        print(
            f"  Perplexity: "
            f"{summary['perplexity']:.4f}"
        )


dev_results = evaluate_file(DEV_FILE)
test_results = evaluate_file(TEST_FILE)


print("=" * 60)
print("LAPLACE SMOOTHING EVALUATION REPORT")
print("=" * 60)

print("\nDEVELOPMENT SET")
print_report("Development Set", dev_results)

print("\n" + "=" * 60)

print("\nTEST SET")
print_report("Test Set", test_results)

LAPLACE SMOOTHING EVALUATION REPORT

DEVELOPMENT SET

Development Set
---------------
Sentences evaluated: 1000

Unigram
  Average log probability: -151.0772
  Perplexity: 409247534791370452918950700728831601301728158072951963853067911168.0000

Bigram
  Average log probability: -171.3000
  Perplexity: 248106987547509997088018750674648136222448090181887560932320539491962978304.0000

Trigram
  Average log probability: -193.6801
  Perplexity: 1300774917967172545361470941570698081776832490900771898318403181145470412343197302784.0000

Quadrigram
  Average log probability: -195.6163
  Perplexity: 9017590727473050071956456035711113594920984667314380172448353553058584174110556815360.0000


TEST SET

Test Set
--------
Sentences evaluated: 1000

Unigram
  Average log probability: -147.1348
  Perplexity: 7940498568517010943715868389258351919830176385544338664492892160.0000

Bigram
  Average log probability: -165.5131
  Perplexity: 761105743249805071194542586504785422527446727042623343397172450047

### Assignment-5 
Apply Add K Smoothing where K=0.3 to all 4 models you developed in the previous assignment
and test it on your test and validation/development sets. Use the following formula for Add K
smoothing.

In [1]:
from pathlib import Path
import duckdb
import math


DB_FILE = Path("ngram_model.duckdb")
DEV_FILE = Path("dev.txt")
TEST_FILE = Path("test.txt")

K = 0.3


con = duckdb.connect(str(DB_FILE), read_only=True)

unigram_counts = {
    row[0]: row[1]
    for row in con.execute(
        "SELECT token, count FROM unigrams"
    ).fetchall()
}

bigram_counts = {
    (row[0], row[1]): row[2]
    for row in con.execute(
        "SELECT w1, w2, count FROM bigrams"
    ).fetchall()
}

trigram_counts = {
    (row[0], row[1], row[2]): row[3]
    for row in con.execute(
        "SELECT w1, w2, w3, count FROM trigrams"
    ).fetchall()
}

quadrigram_counts = {
    (row[0], row[1], row[2], row[3]): row[4]
    for row in con.execute(
        "SELECT w1, w2, w3, w4, count FROM quadrigrams"
    ).fetchall()
}

con.close()


vocab_size = len(unigram_counts)
total_unigrams = sum(unigram_counts.values())


def prepare_sentence(sentence):
    return ["<s>"] + sentence.strip().split() + ["</s>"]


def unigram_log_probability(tokens):
    log_prob = 0.0

    denominator = total_unigrams + K * vocab_size

    for token in tokens:
        count = unigram_counts.get(token, 0)

        probability = (count + K) / denominator

        log_prob += math.log(probability)

    return log_prob


def bigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(1, len(tokens)):
        w1 = tokens[i - 1]
        w2 = tokens[i]

        numerator = bigram_counts.get((w1, w2), 0) + K
        denominator = unigram_counts.get(w1, 0) + K * vocab_size

        probability = numerator / denominator

        log_prob += math.log(probability)

    return log_prob


def trigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(2, len(tokens)):
        w1 = tokens[i - 2]
        w2 = tokens[i - 1]
        w3 = tokens[i]

        numerator = (
            trigram_counts.get((w1, w2, w3), 0) + K
        )

        denominator = (
            bigram_counts.get((w1, w2), 0)
            + K * vocab_size
        )

        probability = numerator / denominator

        log_prob += math.log(probability)

    return log_prob


def quadrigram_log_probability(tokens):
    log_prob = 0.0

    for i in range(3, len(tokens)):
        w1 = tokens[i - 3]
        w2 = tokens[i - 2]
        w3 = tokens[i - 1]
        w4 = tokens[i]

        numerator = (
            quadrigram_counts.get(
                (w1, w2, w3, w4), 0
            ) + K
        )

        denominator = (
            trigram_counts.get(
                (w1, w2, w3), 0
            ) + K * vocab_size
        )

        probability = numerator / denominator

        log_prob += math.log(probability)

    return log_prob


def evaluate_file(file_path):
    sentences = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                sentences.append(
                    prepare_sentence(line)
                )

    results = {
        "sentences": len(sentences),
        "unigram_log_probs": [],
        "bigram_log_probs": [],
        "trigram_log_probs": [],
        "quadrigram_log_probs": []
    }

    for tokens in sentences:
        results["unigram_log_probs"].append(
            unigram_log_probability(tokens)
        )

        results["bigram_log_probs"].append(
            bigram_log_probability(tokens)
        )

        results["trigram_log_probs"].append(
            trigram_log_probability(tokens)
        )

        results["quadrigram_log_probs"].append(
            quadrigram_log_probability(tokens)
        )

    return results


def summarize_model(log_probs):
    average_log_prob = sum(log_probs) / len(log_probs)

    perplexity = math.exp(-average_log_prob)

    return {
        "average_log_probability": average_log_prob,
        "perplexity": perplexity
    }


def print_report(name, results):
    print(f"\n{name}")
    print("-" * len(name))

    print(f"Sentences evaluated: {results['sentences']}")

    models = {
        "Unigram": results["unigram_log_probs"],
        "Bigram": results["bigram_log_probs"],
        "Trigram": results["trigram_log_probs"],
        "Quadrigram": results["quadrigram_log_probs"]
    }

    for model_name, log_probs in models.items():
        summary = summarize_model(log_probs)

        print(f"\n{model_name}")
        print(
            f"  Average log probability: "
            f"{summary['average_log_probability']:.4f}"
        )
        print(
            f"  Perplexity: "
            f"{summary['perplexity']:.4f}"
        )


dev_results = evaluate_file(DEV_FILE)
test_results = evaluate_file(TEST_FILE)


print("=" * 60)
print("ADD-K SMOOTHING EVALUATION REPORT")
print(f"k = {K}")
print("=" * 60)

print("\nDEVELOPMENT SET")
print_report("Development Set", dev_results)

print("\n" + "=" * 60)

print("\nTEST SET")
print_report("Test Set", test_results)

ADD-K SMOOTHING EVALUATION REPORT
k = 0.3

DEVELOPMENT SET

Development Set
---------------
Sentences evaluated: 1000

Unigram
  Average log probability: -151.3563
  Perplexity: 541027910753640032230148556237640847648684388399429889433570115584.0000

Bigram
  Average log probability: -160.3188
  Perplexity: 4222397875339860698632395963141621892909024437943905566927708846292992.0000

Trigram
  Average log probability: -187.5567
  Perplexity: 2850091824659709360163153172596519030190303916395454813581228662207598636625297408.0000

Quadrigram
  Average log probability: -192.6984
  Perplexity: 487382443193444636534597448968610885268397411242468260816687654330248956112552329216.0000


TEST SET

Test Set
--------
Sentences evaluated: 1000

Unigram
  Average log probability: -147.5009
  Perplexity: 11450725983286917975354677897517713561398750305211773826732392448.0000

Bigram
  Average log probability: -155.0871
  Perplexity: 22566830194721643308757196855758390024121244110792014808030904320000